#### RAG pipelines - Data Ingestiion to Vector DB pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

d:\RAG-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## Read all the pdf's inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../Data")

Found 3 PDF files to process

Processing: Batch-norm.pdf
  ✓ Loaded 9 pages

Processing: CNN-for-image-detection.pdf
  ✓ Loaded 5 pages

Processing: CNN-in-comp-vision.pdf
  ✓ Loaded 43 pages

Total documents loaded: 57


In [3]:
all_pdf_documents #DOCUMENT INFORMATION

[Document(metadata={'producer': 'pdfTeX-1.40.14', 'creator': 'LaTeX with hyperref package', 'creationdate': '2015-05-11T13:56:06-07:00', 'author': 'Sergey Ioffe, Christian Szegedy', 'title': 'Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift', 'subject': 'Proceedings of the International Conference on Machine Learning 2015', 'keywords': '', 'moddate': '2015-05-11T13:56:06-07:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.1415926-2.5-1.40.14 (TeX Live 2013/Debian) kpathsea version 6.1.1', 'source': '..\\Data\\pdf\\Batch-norm.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'Batch-norm.pdf', 'file_type': 'pdf'}, page_content='Batch Normalization: Accelerating Deep Network Training by Reducing\nInternal Covariate Shift\nSergey Ioffe SIOFFE @GOOGLE .COM\nChristian Szegedy SZEGEDY @GOOGLE .COM\nGoogle, 1600 Amphitheatre Pkwy, Mountain View, CA 94043\nAbstract\nTraining Deep Neural Networks is complicated

#### Chunkings

In [4]:
### Text splitting get into chunks

def split_documents(documents,chunk_size = 1000,chunk_overlap = 200):
    """split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks = split_documents(all_pdf_documents)
chunks

Split 57 documents into 252 chunks

Example chunk:
Content: Batch Normalization: Accelerating Deep Network Training by Reducing
Internal Covariate Shift
Sergey Ioffe SIOFFE @GOOGLE .COM
Christian Szegedy SZEGEDY @GOOGLE .COM
Google, 1600 Amphitheatre Pkwy, Mou...
Metadata: {'producer': 'pdfTeX-1.40.14', 'creator': 'LaTeX with hyperref package', 'creationdate': '2015-05-11T13:56:06-07:00', 'author': 'Sergey Ioffe, Christian Szegedy', 'title': 'Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift', 'subject': 'Proceedings of the International Conference on Machine Learning 2015', 'keywords': '', 'moddate': '2015-05-11T13:56:06-07:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.1415926-2.5-1.40.14 (TeX Live 2013/Debian) kpathsea version 6.1.1', 'source': '..\\Data\\pdf\\Batch-norm.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'Batch-norm.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.14', 'creator': 'LaTeX with hyperref package', 'creationdate': '2015-05-11T13:56:06-07:00', 'author': 'Sergey Ioffe, Christian Szegedy', 'title': 'Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift', 'subject': 'Proceedings of the International Conference on Machine Learning 2015', 'keywords': '', 'moddate': '2015-05-11T13:56:06-07:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.1415926-2.5-1.40.14 (TeX Live 2013/Debian) kpathsea version 6.1.1', 'source': '..\\Data\\pdf\\Batch-norm.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'Batch-norm.pdf', 'file_type': 'pdf'}, page_content='Batch Normalization: Accelerating Deep Network Training by Reducing\nInternal Covariate Shift\nSergey Ioffe SIOFFE @GOOGLE .COM\nChristian Szegedy SZEGEDY @GOOGLE .COM\nGoogle, 1600 Amphitheatre Pkwy, Mountain View, CA 94043\nAbstract\nTraining Deep Neural Networks is complicated

### Embedding

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"): ## Model name avaiable in Huggingface
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model() ## will load function

    def _load_model(self): ## protected function
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray: # returns numpy array
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    




## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2745.92it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


C:\Users\hb292\AppData\Local\Temp\ipykernel_26736\3408326387.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../Data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 252


In [9]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.14', 'creator': 'LaTeX with hyperref package', 'creationdate': '2015-05-11T13:56:06-07:00', 'author': 'Sergey Ioffe, Christian Szegedy', 'title': 'Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift', 'subject': 'Proceedings of the International Conference on Machine Learning 2015', 'keywords': '', 'moddate': '2015-05-11T13:56:06-07:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.1415926-2.5-1.40.14 (TeX Live 2013/Debian) kpathsea version 6.1.1', 'source': '..\\Data\\pdf\\Batch-norm.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'Batch-norm.pdf', 'file_type': 'pdf'}, page_content='Batch Normalization: Accelerating Deep Network Training by Reducing\nInternal Covariate Shift\nSergey Ioffe SIOFFE @GOOGLE .COM\nChristian Szegedy SZEGEDY @GOOGLE .COM\nGoogle, 1600 Amphitheatre Pkwy, Mountain View, CA 94043\nAbstract\nTraining Deep Neural Networks is complicated

In [10]:
##extract all the text from particular chunks and generate a embedding

texts = [doc.page_content for doc in chunks]


## generate embedding 
embeddings = embedding_manager.generate_embeddings(texts)

## store in the vectordatabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 252 texts...


Batches: 100%|██████████| 8/8 [00:08<00:00,  1.10s/it]


Generated embeddings with shape: (252, 384)
Adding 252 documents to vector store...
Successfully added 252 documents to vector store
Total documents in collection: 504


### retriever Pipeline from VectorStore

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance ##how similar the data comming out 
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [13]:
rag_retriever

In [ ]:
rag_retriever.retrieve("what is CNN")

#this is context

Retrieving documents for query: 'what is CNN'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.63it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_aefeb450_94',
  'content': 'fied and presented.\nThe remaining part of the paper proceeds as follows (shown in Fig.\xa0 1): Section\xa0 2 gives \na basic introduction to the elementary components of CNN and their corresponding \nfunctions. Sections\xa0 3, 4, and 5  summarize the relevant research models and methods in \nthree application directions, namely, image classification, object detection, and video \nprediction, respectively. In Sects. 6  and 7 , through synthesizing the current research \nstatus, the issues of CNN are analyzed and summarized. In addition, an outlook on \nfuture research trends is provided.\n2  Basic CNN components\nAlthough there are numerous variations of CNN models, the overall architecture is essen-\ntially the same and adheres to a fixed paradigm, consisting of an input layer, alternate lay -\ners of convolution and pooling layers, one or more fully connected layers, activation func-\ntions, and an output layer at the end. The first half of th

In [ ]:
rag_retriever.retrieve("Training and Inference with Batch-Normalized Networks")
#response comming from vectorstore

Retrieving documents for query: 'Training and Inference with Batch-Normalized Networks'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 30.84it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_4a0aaaac_46',
  'content': 'propagate the gradients through the normalization param-\neters. Batch Normalization adds only two extra parame-\nters per activation, and in doing so preserves the represen-\ntation ability of the network. We presented an algorithm\nfor constructing, training, and performing inference with\nbatch-normalized networks. The resulting networks can be\ntrained with saturating nonlinearities, are more tolerant to\nincreased training rates, and often do not require Dropout\nfor regularization.\nMerely adding Batch Normalization to a state-of-the-art\nimage classiﬁcation model yields a substantial speedup in\ntraining. By further increasing the learning rates, remov-\ning Dropout, and applying other modiﬁcations afforded by\nBatch Normalization, we reach the previous state of the\nart with only a small fraction of training steps – and then\nbeat the state of the art in single-network image classiﬁca-\ntion. Furthermore, by combining multiple models tra

### Query Retrieval Pipeline(Augmented Generation)


query-->vector-->context+prompt-->LLM-->o/p

In [23]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

# initialize the Groq LLM
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content


In [24]:
answer = rag_simple("what is CNN?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'what is CNN?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 53.76it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


CNN stands for Convolutional Neural Network.


### Enchanced RAG Pipeline

In [29]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:400] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke(prompt)
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("what is CNN", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'what is CNN'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 47.05it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: CNN stands for Convolutional Neural Network.
Sources: [{'source': 'CNN-in-comp-vision.pdf', 'page': 2, 'score': 0.33215880393981934, 'preview': 'fied and presented.\nThe remaining part of the paper proceeds as follows (shown in Fig.\xa0 1): Section\xa0 2 gives \na basic introduction to the elementary components of CNN and their corresponding \nfunctions. Sections\xa0 3, 4, and 5  summarize the relevant research models and methods in \nthree application directions, namely, image classification, object detection, and video \nprediction, respectively. In S...'}, {'source': 'CNN-in-comp-vision.pdf', 'page': 2, 'score': 0.33215880393981934, 'preview': 'fied and presented.\nThe remaining part of the paper proceeds as follows (shown in Fig.\xa0 1): Section\xa0 2 gives \na basic introduction to the elementary components of CNN and their corresponding \nfunctions. Sections\xa0 3, 4, and 5  summarize the relevant research models and methods in \nthree application directions, namely, imag

In [31]:
# # --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
# from typing import List, Dict, Any
# import time

# class AdvancedRAGPipeline:
#     def __init__(self, retriever, llm):
#         self.retriever = retriever
#         self.llm = llm
#         self.history = []  # Store query history

#     def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
#         # Retrieve relevant documents
#         results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
#         if not results:
#             answer = "No relevant context found."
#             sources = []
#             context = ""
#         else:
#             context = "\n\n".join([doc['content'] for doc in results])
#             sources = [{
#                 'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
#                 'page': doc['metadata'].get('page', 'unknown'),
#                 'score': doc['similarity_score'],
#                 'preview': doc['content'][:120] + '...'
#             } for doc in results]
#             # Streaming answer simulation
#             prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
#             if stream:
#                 print("Streaming answer:")
#                 for i in range(0, len(prompt), 80):
#                     print(prompt[i:i+80], end='', flush=True)
#                     time.sleep(0.05)
#                 print()
#             response = self.llm.invoke([prompt.format(context=context, question=question)])
#             answer = response.content

#         # Add citations to answer
#         citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
#         answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

#         # Optionally summarize answer
#         summary = None
#         if summarize and answer:
#             summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
#             summary_resp = self.llm.invoke([summary_prompt])
#             summary = summary_resp.content

#         # Store query history
#         self.history.append({
#             'question': question,
#             'answer': answer,
#             'sources': sources,
#             'summary': summary
#         })

#         return {
#             'question': question,
#             'answer': answer_with_citations,
#             'sources': sources,
#             'summary': summary,
#             'history': self.history
#         }

# # Example usage:
# adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
# result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
# print("\nFinal Answer:", result['answer'])
# print("Summary:", result['summary'])
# print("History:", result['history'][-1])